# 📡 Module 3 — Class 5: Data Pipelines

**A Pipeline = one object that does ALL the work: clean → scale → encode → predict.**

---

## 📖 Today's story

You work as a data analyst at a telecom company in Tashkent.
Your manager comes to your desk and says:

> "We have 7,000 customers. Many are leaving us every month (we call this *churn*).
> Build a model that tells me WHO will leave next month."

You spent 4 classes (M3 C1, C2, C3, C4) learning:

- C1 — explore data, find missing values
- C2 — scale numbers (StandardScaler, MinMaxScaler)
- C3 — encode text categories (OneHotEncoder)
- C4 — select important features

You build the model. It works. You show it to your manager. He smiles.

Tomorrow he comes back and says:

> "Good. Now put this in **production**. When a new customer signs up today,
> do the SAME cleaning and SAME scaling and predict churn. Show me the code."

You panic. Why?

1. Your code is in 4 different notebooks.
2. You have 4 fitted objects (median, scaler, encoder, model). You must save each one.
3. If you forget ONE step in production, the model gives wrong predictions silently.
4. A new teammate will not understand the right order.

**Today we solve this. We wrap EVERYTHING into one `Pipeline` object.**
**One `.joblib` file → ready for production.**

---

## 🎯 Today's plan (about 45 minutes)

| Time | Topic |
| --- | --- |
| 5 min | Why we need pipelines (2 problems) |
| 5 min | Load real Telco data + pick 5 columns |
| 10 min | The OLD way (manual code) — feel the pain |
| 10 min | First simple Pipeline (2 steps) |
| 10 min | ColumnTransformer (different rules for different columns) |
| 3 min | Full Pipeline = preprocessor + model |
| 2 min | Save with `joblib.dump`, load with `joblib.load` |

---

## 🗺️ Where we are in Module 3

| Class | Topic | Status |
| --- | --- | --- |
| C1 | EDA + missing values | ✅ done |
| C2 | Scaling (StandardScaler, MinMaxScaler) | ✅ done |
| C3 | Encoding (OneHotEncoder, LabelEncoder) | ✅ done |
| C4 | Feature Selection (Correlation, VIF, RFE) | ✅ done |
| **C5** | **Pipelines — today** | 🎯 |
| C6 | Lab — combine everything for the project | 🚧 next |

Today's class connects ALL previous classes into ONE object. After today,
your code is short, clean, and ready for production.

---

## 🚨 The 2 problems a Pipeline solves

### Problem 1: Data leakage

**Wrong way (most beginners do this):**

1. Fit `StandardScaler` on the FULL dataset (train + test together).
2. Then split into train / test.

Why is this wrong?

The scaler learned the MEAN of the test data.
The model now has secret information about the test set.
Test accuracy looks high. But in production it crashes.

**Right way:**

1. Split FIRST.
2. Fit scaler ONLY on training data.
3. Apply the SAME scaler to the test data (no re-fit).

A Pipeline does this automatically. It **cannot** peek at test data.

### Problem 2: Train / serve skew

**Wrong way:**

- For training: clean data in your notebook.
- For production: write different cleaning code in the API.

Why is this wrong?

Two pieces of code never stay equal forever.
One day you fix a bug in the training code. You forget the production code.
Model accuracy drops. No error message. Silent bug. Manager angry.

**Right way:**

- ONE Pipeline object.
- ONE saved file (`.joblib`).
- The SAME `pipeline.predict()` runs in training and in production.

---

## 📦 Setup — import libraries

We need 8 tools from `sklearn` plus `pandas`, `numpy`, and `joblib` today.

In [ ]:
# pandas — for tables (DataFrame). Similar to an Excel sheet inside Python.
import pandas as pd

# numpy — for numeric arrays. Used behind pandas for math.
import numpy as np

# Pipeline — the BIG one today. Chains preprocessing steps + model into one object.
from sklearn.pipeline import Pipeline

# ColumnTransformer — sends different columns through different mini-pipelines.
from sklearn.compose import ColumnTransformer

# StandardScaler — makes numbers have mean=0 and std=1 (from C2).
from sklearn.preprocessing import StandardScaler

# OneHotEncoder — turns text categories into 0/1 columns (from C3).
from sklearn.preprocessing import OneHotEncoder

# SimpleImputer — fills missing values (with median, mean, or most-frequent).
from sklearn.impute import SimpleImputer

# LogisticRegression — simple Yes / No model. We need a model at the end of the pipeline.
from sklearn.linear_model import LogisticRegression

# train_test_split — splits the data into a training part and a testing part.
from sklearn.model_selection import train_test_split

# accuracy_score — counts how many predictions are correct.
# classification_report — prints precision / recall / f1 per class.
from sklearn.metrics import accuracy_score, classification_report

# joblib — saves a trained pipeline to a file (.joblib).
import joblib

# Print a confirmation so we know the cell ran.
print("✅ All tools imported")

---

## 📂 Load the real Telco Customer Churn dataset

This is a real dataset from IBM. **7,043 telecom customers.**
Each row = one customer. The target column is `Churn` (Yes = left, No = stayed).
We read it directly from GitHub — no download needed.

In [ ]:
# URL of the public IBM Telco Customer Churn CSV file.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# pd.read_csv downloads the CSV and parses it into a DataFrame.
df = pd.read_csv(url)

# .shape gives (rows, columns). We expect (7043, 21).
print("Shape:", df.shape)

# .columns shows all column names so we can pick the ones we want later.
print("\nColumns:")
print(list(df.columns))

# .head() shows the first 5 rows.
df.head()

---

## 🧹 One tiny clean before we start

`TotalCharges` looks numeric but Python reads it as text (object).
A few rows have empty spaces instead of numbers. We must fix this BEFORE the pipeline,
because the column type itself is wrong (this is not normal cleaning — it is a type fix).

In [ ]:
# pd.to_numeric converts a column to numbers.
# errors='coerce' = if a cell can't become a number, set it to NaN (missing).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# .isna() returns True/False per cell. .sum() counts the True values.
# We only check TotalCharges to see how many cells we converted to missing.
print("Missing TotalCharges after type fix:", df['TotalCharges'].isna().sum())

# Show the unique values of the target column. We expect Yes / No.
print("Churn classes:", df['Churn'].unique())

---

## 🎯 Pick 5 features + the target

To keep the class simple we use only 5 features:

| Column | Type | Meaning |
| --- | --- | --- |
| `tenure` | numeric | how long customer has been with us (months) |
| `MonthlyCharges` | numeric | how much customer pays per month |
| `TotalCharges` | numeric | total money customer paid so far |
| `Contract` | categorical | Month-to-month / One year / Two year |
| `PaymentMethod` | categorical | Electronic check / Mailed check / Bank transfer / Credit card |

**Target:** `Churn` (Yes = customer left, No = customer stayed)

In [ ]:
# numeric_cols — list of columns that hold numbers.
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# categorical_cols — list of columns that hold text categories.
categorical_cols = ['Contract', 'PaymentMethod']

# target_col — what we want to predict.
target_col = 'Churn'

# X = features (input). We pick only the 5 columns we chose.
X = df[numeric_cols + categorical_cols]

# y = target. We map Yes → 1 and No → 0 so the model can handle it.
y = df[target_col].map({'Yes': 1, 'No': 0})

# Print to confirm.
print("X shape:", X.shape)
print("y mean (= churn rate):", round(y.mean(), 3))
print("\nAbout 27% of customers churn. This is a moderate imbalance — Logistic Regression can handle it.")

---

## ✂️ Train / test split

We MUST split BEFORE any preprocessing. This is the leakage rule.

- `test_size=0.2` → 80% train, 20% test
- `random_state=42` → same split every time we run the code (reproducibility)
- `stratify=y` → keep the SAME churn ratio (≈27%) in train and test

In [ ]:
# train_test_split splits X and y at the same time so rows stay aligned.
# Returns 4 things, in this exact order: X_train, X_test, y_train, y_test.
X_train, X_test, y_train, y_test = train_test_split(
    X,                # features
    y,                # target
    test_size=0.2,    # 20% goes to test set
    random_state=42,  # fixed seed → same split every time
    stratify=y,       # keep the churn rate equal in train and test
)

# Always print shapes after a split. Catch mistakes early.
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 3))
print("Test  churn rate:", round(y_test.mean(), 3))

---

## 😩 Section A — The OLD way (no Pipeline)

Before we see the Pipeline, let's feel the PAIN of doing everything manually.
We need 5 steps. In the right order. No leakage.

The code below is INTENTIONALLY long. We will **not** use it after this section.
We want to see why Pipelines are good.

In [ ]:
# Step 1: Fill the 11 missing TotalCharges with the MEDIAN.
# We learn the median from TRAIN only (no leakage).
median_total = X_train['TotalCharges'].median()                # learn from train
X_train_m = X_train.copy()                                     # copy so we don't mutate original
X_test_m  = X_test.copy()
X_train_m['TotalCharges'] = X_train_m['TotalCharges'].fillna(median_total)  # apply to train
X_test_m['TotalCharges']  = X_test_m['TotalCharges'].fillna(median_total)   # apply SAME median to test

# Step 2: Scale numeric columns to mean=0, std=1.
scaler = StandardScaler()                                      # create scaler
scaler.fit(X_train_m[numeric_cols])                            # learn mean/std from TRAIN
X_train_num = scaler.transform(X_train_m[numeric_cols])        # apply to train
X_test_num  = scaler.transform(X_test_m[numeric_cols])         # apply SAME numbers to test

# Step 3: One-hot encode categorical columns.
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train_m[categorical_cols])                       # learn categories from TRAIN
X_train_cat = encoder.transform(X_train_m[categorical_cols])   # apply to train
X_test_cat  = encoder.transform(X_test_m[categorical_cols])    # apply SAME categories to test

# Step 4: Glue numeric + categorical back into one matrix.
X_train_final = np.hstack([X_train_num, X_train_cat])          # horizontal stack (columns)
X_test_final  = np.hstack([X_test_num,  X_test_cat])

# Step 5: Train the model.
model_old = LogisticRegression(max_iter=1000)                  # max_iter=1000 to make sure it converges
model_old.fit(X_train_final, y_train)                          # learn weights

# Score on test set.
acc_old = model_old.score(X_test_final, y_test)
print(f"Old-way accuracy: {acc_old:.3f}")
print()
print("⚠️ Look at this code. 25+ lines. 4 separate fitted objects (median, scaler, encoder, model).")
print("⚠️ Each one must be SAVED separately. Each one must be LOADED in production.")
print("⚠️ If you forget ONE step in production, the model breaks silently.")

---

## ✨ Section B — The NEW way: `Pipeline`

A `Pipeline` is a chain of steps. Each step has a **name** and an **object**.
The LAST step is usually a model. All other steps must have `.fit()` and `.transform()` methods.

When we call `pipeline.fit(X, y)`:

1. The first step runs `.fit_transform(X)`.
2. The result passes to the next step.
3. … until the last step (model) runs `.fit(X, y)`.

When we call `pipeline.predict(X_new)`:

1. The first step runs `.transform(X_new)` — NOT fit. It uses values it already learned.
2. The result passes to the next step.
3. … until the last step (model) runs `.predict()`.

This is exactly why pipelines prevent leakage by design.

### First simple example — only numeric columns

Pipeline takes a LIST of `(name, object)` tuples. We start with 2 steps.

In [ ]:
# Tiny example: a pipeline for ONLY the numeric columns.
simple_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # step 1 — fill missing with median
    ('scaler',  StandardScaler()),                   # step 2 — scale to mean=0, std=1
])

# .fit_transform learns + applies in one call (only call this on TRAIN data).
X_train_num_v2 = simple_pipe.fit_transform(X_train[numeric_cols])

# .transform only applies (it uses values learned during the fit above).
# Always call .transform on test data — NEVER .fit_transform.
X_test_num_v2 = simple_pipe.transform(X_test[numeric_cols])

# Check the shape and the first row.
print("X_train_num_v2 shape:", X_train_num_v2.shape)
print("First row of train (scaled):", X_train_num_v2[0].round(3))
print()
print("✅ 2 steps replaced 6 lines of manual code (median + fillna + scaler.fit + transform).")

---

## 🔀 Section C — `ColumnTransformer`

Numeric and categorical columns need DIFFERENT preprocessing:

- **Numeric** → impute median + scale
- **Categorical** → impute most-frequent + one-hot encode

We cannot use ONE simple Pipeline for both.
We need `ColumnTransformer` to ROUTE different columns to different mini-pipelines.

In [ ]:
# Mini-pipeline #1 — only for NUMERIC columns.
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # fill missing with median
    ('scaler',  StandardScaler()),                   # scale to mean=0, std=1
])

# Mini-pipeline #2 — only for CATEGORICAL columns.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),   # fill missing with the mode (most common)
    ('onehot',  OneHotEncoder(handle_unknown='ignore',      # in production, ignore unknown categories
                              sparse_output=False)),        # return a normal numpy array, not sparse
])

# ColumnTransformer glues them together.
# .transformers takes a LIST of (name, mini_pipeline, columns) tuples.
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     numeric_cols),       # apply mini-pipeline #1 to numeric_cols
    ('cat', categorical_transformer, categorical_cols),   # apply mini-pipeline #2 to categorical_cols
])

print("✅ ColumnTransformer built")
print(f"   numeric_transformer     → {numeric_cols}")
print(f"   categorical_transformer → {categorical_cols}")

---

## 🧪 Quick test — run the preprocessor alone (no model yet)

We want to SEE what the preprocessor does. The output is a numeric matrix.

In [ ]:
# .fit_transform on training data: learn medians + means + categories, then transform.
X_train_prep = preprocessor.fit_transform(X_train)

# .transform on test data: apply ALREADY-LEARNED values (no peek at test).
X_test_prep = preprocessor.transform(X_test)

# Show shapes. Numeric columns keep their count. Categorical columns expand via OneHot.
print("X_train_prep shape:", X_train_prep.shape)
print("Original column count:", X_train.shape[1])
print("After preprocessing:  ", X_train_prep.shape[1])
print()
print("Extra columns come from OneHot expanding `Contract` (3 categories)")
print("and `PaymentMethod` (4 categories) → +5 columns total.")
print()
print("First processed row (numbers + 0/1 dummies):")
print(X_train_prep[0].round(2))

---

## 🏗️ Section D — Full Pipeline: preprocessor + model

This is the magic moment. We add the model as the LAST step of the pipeline.
Now ONE object does EVERYTHING: impute → scale → encode → predict.

In [ ]:
# Full pipeline = preprocessor (ColumnTransformer) + model.
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),                              # step 1 — clean + scale + encode
    ('model',        LogisticRegression(max_iter=1000,           # step 2 — predict 0/1
                                        random_state=42)),
])

# We don't call .fit_transform here — just .fit
# (because the LAST step is a model, not a transformer).
full_pipeline.fit(X_train, y_train)

print("✅ Pipeline trained in ONE line of code:")
print("   full_pipeline.fit(X_train, y_train)")

---

## 🎯 Predict + score

`.predict(X_test)` runs ALL steps in order: impute → scale → encode → model.

In [ ]:
# .predict applies all steps to X_test and returns 0/1 predictions.
y_pred = full_pipeline.predict(X_test)

# Accuracy = (correct predictions) / (total predictions).
acc_new = accuracy_score(y_test, y_pred)
print(f"Pipeline accuracy:  {acc_new:.3f}")
print(f"Old-way accuracy:   {acc_old:.3f}")
print()
print("Same accuracy. Much cleaner code. Production-ready.")

# classification_report shows precision, recall, f1 per class.
print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=['No churn', 'Churn']))

---

## 💾 Section E — Save the pipeline

`joblib.dump` writes the trained pipeline to a file.
Production server loads this file. No code copying. No leakage. No bugs.

In [ ]:
# joblib.dump(object, filename) saves the object to disk.
# .joblib is the standard file extension for sklearn objects.
joblib.dump(full_pipeline, 'telco_churn_pipeline.joblib')

print("✅ Saved telco_churn_pipeline.joblib")
print()
print("Production server only needs three things:")
print("   1. Python + sklearn installed")
print("   2. The file telco_churn_pipeline.joblib")
print("   3. One line of code:  prediction = pipeline.predict(new_customer)")

---

## 📦 Section F — Load + predict on a NEW customer

This is what production code looks like.

In [ ]:
# joblib.load reads the file back into a Pipeline object (identical to before).
loaded_pipeline = joblib.load('telco_churn_pipeline.joblib')

# Pretend a new customer just signed up. Build ONE row as a DataFrame.
# Column names MUST match the columns we trained on.
new_customer = pd.DataFrame([{
    'tenure':          5,                  # 5 months with us
    'MonthlyCharges':  85.0,               # pays 85 USD per month
    'TotalCharges':    425.0,              # paid 425 USD total so far
    'Contract':        'Month-to-month',   # risky contract type
    'PaymentMethod':   'Electronic check', # most-churn-prone payment method
}])

# .predict returns an array. We take [0] = the first (and only) row.
prediction = loaded_pipeline.predict(new_customer)[0]

# .predict_proba returns probability per class. [0, 1] = row 0, class 1 (churn).
probability = loaded_pipeline.predict_proba(new_customer)[0, 1]

print("Prediction:", "Churn" if prediction == 1 else "No churn")
print(f"Churn probability: {probability:.1%}")

---

## 🔒 Section G — Why pipelines prevent leakage (by design)

When you call `full_pipeline.fit(X_train, y_train)`, INSIDE the pipeline:

1. The `preprocessor` calls `.fit_transform(X_train)`:
   - `SimpleImputer` learns medians ONLY from X_train
   - `StandardScaler` learns mean/std ONLY from X_train
   - `OneHotEncoder` learns categories ONLY from X_train
2. The `model.fit()` runs on the transformed X_train.

When you call `full_pipeline.predict(X_test)`:

1. The `preprocessor` calls `.transform(X_test)` — **not fit**. Uses values from step 1.
2. The `model.predict()` runs on the transformed X_test.

The pipeline does **not** have permission to "look at" X_test during training.
This is mathematically impossible to do leakage.

🎯 **Pipelines = leakage prevention by design.**

---

## ⚠️ Common mistakes

### Mistake 1 — Fit the preprocessor BEFORE splitting

```python
# WRONG — peeks at test data
scaler.fit(X)
X_train, X_test = train_test_split(X)
```

Use a Pipeline. Then this mistake is impossible.

### Mistake 2 — Forget `handle_unknown='ignore'`

In production, a new customer may have a category we never saw in training.
Default behaviour → `OneHotEncoder` crashes with an error.
With `handle_unknown='ignore'` → it returns a row of zeros and the model still runs.

### Mistake 3 — Use `sparse_output=True` (the default)

Sparse matrices break some downstream code. Always set `sparse_output=False` unless your data has millions of rows.

### Mistake 4 — Mix column names

DataFrame columns must have the SAME NAMES in training and in production.
If production sends `monthly_charges` but you trained with `MonthlyCharges`, you get an error.

---

## 🎁 Bonus — inspect the final column names

After fitting, you can ask the pipeline what the final columns are called.
This is useful when you want to see feature importance later.

In [ ]:
# .get_feature_names_out() returns the column names AFTER all transformations.
# We call it on the preprocessor (the ColumnTransformer step inside the pipeline).
final_columns = full_pipeline.named_steps['preprocessor'].get_feature_names_out()

print(f"Total columns after preprocessing: {len(final_columns)}")
print()
print("Column names:")
for i, name in enumerate(final_columns):
    print(f"  {i:2d}.  {name}")

---

## ✅ Recap

| Tool | Job |
| --- | --- |
| `Pipeline` | Chain of steps. Last step is a model. |
| `ColumnTransformer` | Route different columns to different mini-pipelines. |
| `SimpleImputer` | Fill missing values (median / most_frequent). |
| `StandardScaler` | Scale numbers (mean=0, std=1). |
| `OneHotEncoder` | Convert text categories to 0/1 columns. |
| `joblib.dump` / `.load` | Save and load trained pipelines. |

### 📝 Words to remember

- **Leakage** — when test data influences training. Bad.
- **Train / serve skew** — different cleaning code in training vs production. Bad.
- **Fit** — learn medians / means / categories from TRAINING data.
- **Transform** — apply learned values to NEW data.
- **`handle_unknown='ignore'`** — safe choice for production: ignore unknown categories.

### 🎯 Big idea

> ONE pipeline. ONE file. SAME `.predict()` in training and production.

---

## 📤 Homework

### Bronze
1. Add two more columns to the pipeline: `gender` and `SeniorCitizen`.
2. Retrain the pipeline. Report the new accuracy.

### Silver
3. Try `RandomForestClassifier` (from `sklearn.ensemble`) instead of `LogisticRegression`. Compare accuracy.
4. Save the new pipeline as `telco_churn_rf.joblib`.

### Gold
5. Build a small `predict_one(customer_dict)` function that:
   - takes a Python dict like `{'tenure': 5, 'Contract': 'Month-to-month', ...}`
   - converts it to a DataFrame
   - calls the loaded pipeline
   - returns the probability of churn

🎯 Goal: a function we could drop into a FastAPI endpoint tomorrow.